In [ ]:
import geopandas as gpd
import xarray as xr
import xvec  # noqa: F401

In [ ]:
# eez_geoparquet_path: s3://csdr-dev-env-data-ap-southeast-2/EEZ_land_union_v4_202410.parquet
# aca_zarr_path: s3://csdr-dev-env-data-ap-southeast-2/aca.zarr
ds = xr.open_zarr("s3://csdr-dev-env-data-ap-southeast-2/aca.zarr")

In [ ]:
# bbox over SE Asia, including Australia
bbox = [95.0, -11.0, 160.0, 10.0]

gdf = gpd.read_parquet(
    "s3://csdr-dev-env-data-ap-southeast-2/EEZ_land_union_v4_202410.parquet"
)
gdf.head()

In [ ]:
# Filter the GeoDataFrame to only include geometries that intersect with the bounding box
gdf_filtered = gdf.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]]
gdf_filtered.head()

In [ ]:
da = ds["reef_mask"]

In [ ]:
stats_da = da.xvec.zonal_stats(
    gdf_filtered[0:1].geometry,  # Can't even do it with 1 geometry
    x_coords="lon",
    y_coords="lat",
    stats=["sum"],
    # method="exactextract"
)

pixel_area = abs(stats_da.rio.resolution()[0] * stats_da.rio.resolution()[1])
area_da = stats_da.sel(zonal_statistics="sum" * pixel_area)
stats_da.loc[dict(zonal_statistics="sum")] = area_da.values

stats_computed = stats_da.compute()  # Compute the dask array